# Phase 2 — Exploratory Data Analysis (EDA)
## YouTube Quality Analyzer — Corpus de commentaires

**Objectif** : Explorer et valider le dataset CSV avant de lancer le pipeline multi-agents.

**Livrables attendus** :
- Statistiques descriptives du corpus
- Distribution des longueurs de commentaires
- Distribution des langues détectées
- Distribution des métriques d’engagement (author_likes, reply_count)
- Identification des signaux de bruit évidents
- Export d’un aperçu nettoyé vers `data/clean/`

**Tag Git cible** : `v0.1.0`

---
## 0. Configuration

In [3]:
import random
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproductibilité — NFR-02 (seed fixé à 42)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Chemins
CSV_PATH = "../data/raw/comments_raw.csv"
CLEAN_OUTPUT = "../data/clean/comments_eda_preview.csv"

# Colonnes attendues (PRD §4.2 — A1 Loader)
REQUIRED_COLS = ["text"]
OPTIONAL_COLS = ["video_id", "author_likes", "reply_count"]

print("Configuration OK — seed =", SEED)

Configuration OK — seed = 42


---
## 1. Chargement et validation du CSV

In [4]:
df = pd.read_csv(CSV_PATH)
print(f"Shape : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"Colonnes : {list(df.columns)}")
df.head()

Shape : 42860 lignes x 6 colonnes
Colonnes : ['commentaire_id', 'video_id', 'texte_commentaire', 'publie_le', 'nb_likes_commentaire', 'nb_reponses']


,commentaire_id,video_id,texte_commentaire,publie_le,nb_likes_commentaire,nb_reponses
0,UgxoD16zA6d6P-Gu_jp4AaABAg,2dwMa7aeb0M,Moi je suis nulle en maths mais grâce à vos vi...,2025-10-12T17:56:42Z,9,0
1,UghvGEzou_Ana3gCoAEC,2dwMa7aeb0M,grâce a vous je suis devenu plus fort en maths...,2016-04-21T16:36:18Z,22,0
2,UgxAugZhHsUnjyRqWaZ4AaABAg,2dwMa7aeb0M,je suis en 3e et heureuse d&#39;avoir découver...,2018-03-29T19:26:40Z,38,6
3,Ugg0yvVZvm9KangCoAEC,2dwMa7aeb0M,J&#39;ai toujours eu beaucoup de mal en mathém...,2016-09-23T19:09:38Z,55,5
4,Ugjhos2JSgNa1XgCoAEC,2dwMa7aeb0M,"Merci bcp, les explications sont claires, ça a...",2016-05-29T17:50:28Z,27,0


In [7]:
missing_required = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_required:
    raise ValueError(f"Colonnes requises manquantes : {missing_required}")

present_optional = [c for c in OPTIONAL_COLS if c in df.columns]
missing_optional = [c for c in OPTIONAL_COLS if c not in df.columns]
print(f"OK Colonnes requises     : {REQUIRED_COLS}")
print(f"OK Colonnes opt. présentes : {present_optional}")
if missing_optional:
    print(f"Warning colonnes opt. absentes : {missing_optional}")

ValueError: Colonnes requises manquantes : ['text']

In [ ]:
print("Types :")
print(df.dtypes.to_string())
print("\nValeurs nulles :")
print(df.isnull().sum().to_string())

---
## 2. Statistiques descriptives

In [ ]:
df_clean = df.copy()
df_clean["text"] = df_clean["text"].astype(str).str.strip()
df_clean = df_clean[df_clean["text"].str.len() > 0].copy()

df_clean["char_length"] = df_clean["text"].str.len()
df_clean["word_count"]  = df_clean["text"].str.split().str.len()

print(f"Après suppression vides : {len(df_clean)} (supprimés : {len(df) - len(df_clean)})")
print()
print(df_clean[["char_length", "word_count"]].describe().round(2).to_string())

In [ ]:
single_word = df_clean[df_clean["word_count"] == 1]
print(f"Commentaires 1 mot : {len(single_word)} ({100*len(single_word)/len(df_clean):.1f} %)")
print(single_word["text"].value_counts().head(10).to_string())

long_comments = df_clean[df_clean["word_count"] > 100]
print(f"\nCommentaires >100 mots : {len(long_comments)} ({100*len(long_comments)/len(df_clean):.1f} %)")

---
## 3. Distribution des longueurs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_clean["char_length"].clip(upper=500), bins=50,
             color="steelblue", edgecolor="white")
axes[0].set_title("Longueur en caractères (cap 500)")
axes[0].set_xlabel("Caractères")
axes[0].set_ylabel("Nb commentaires")
axes[0].axvline(3, color="red", linestyle="--", label="Min A2 (3 chars)")
axes[0].legend(fontsize=8)

axes[1].hist(df_clean["word_count"].clip(upper=100), bins=50,
             color="teal", edgecolor="white")
axes[1].set_title("Nombre de mots (cap 100)")
axes[1].set_xlabel("Mots")
axes[1].set_ylabel("Nb commentaires")
axes[1].axvline(100, color="red", linestyle="--", label="Seuil long (Musleh 2023)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("../docs/phase2_data/fig_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé : docs/phase2_data/fig_length_distribution.png")

---
## 4. Détection des langues

In [ ]:
def safe_detect(text):
    try:
        from langdetect import detect
        return detect(str(text))
    except Exception:
        return "unknown"

sample_size = min(500, len(df_clean))
df_sample = df_clean.sample(sample_size, random_state=SEED).copy()
print(f"Détection de langue sur {sample_size} commentaires...")
df_sample["language"] = df_sample["text"].apply(safe_detect)
lang_counts = df_sample["language"].value_counts()
print(lang_counts.head(10).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
top_langs = lang_counts.head(10)
bars = ax.barh(top_langs.index[::-1], top_langs.values[::-1], color="steelblue")
ax.set_title("Top 10 langues détectées (500 commentaires)")
ax.set_xlabel("Nb commentaires")
for bar, val in zip(bars, top_langs.values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val} ({100*val/sample_size:.1f}%)", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("../docs/phase2_data/fig_language_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Métriques d’engagement

In [ ]:
engagement_cols = [c for c in ["author_likes", "reply_count"] if c in df_clean.columns]

if engagement_cols:
    print(df_clean[engagement_cols].describe().round(2).to_string())

    fig, axes = plt.subplots(1, len(engagement_cols), figsize=(6 * len(engagement_cols), 4))
    if len(engagement_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, engagement_cols):
        cap = df_clean[col].quantile(0.95)
        ax.hist(df_clean[col].dropna().clip(upper=cap), bins=40, color="coral", edgecolor="white")
        ax.set_title(f"{col} (cap 95e percentile)")
        ax.set_xlabel(col)
        ax.set_ylabel("Nb commentaires")
        med = df_clean[col].median()
        ax.axvline(med, color="navy", linestyle="--", label=f"Médiane: {med:.0f}")
        ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig("../docs/phase2_data/fig_engagement_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Warning : author_likes et reply_count absents du CSV.")

---
## 6. Analyse par vidéo

In [ ]:
if "video_id" in df_clean.columns:
    video_counts = df_clean["video_id"].value_counts()
    print(f"Vidéos distinctes : {df_clean['video_id'].nunique()}")
    print(f"Commentaires/vidéo — moy : {video_counts.mean():.0f}, méd : {video_counts.median():.0f}")
    print(video_counts.head(10).to_frame("nb_commentaires").to_string())

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(video_counts, bins=30, color="mediumpurple", edgecolor="white")
    ax.set_title("Distribution du nb de commentaires par vidéo")
    ax.set_xlabel("Nb commentaires")
    ax.set_ylabel("Nb vidéos")
    plt.tight_layout()
    plt.savefig("../docs/phase2_data/fig_comments_per_video.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Warning : colonne video_id absente.")

---
## 7. Heuristiques de bruit pré-pipeline

Ces flags sont des estimations rapides **avant** le passage par A5 (Noise Detector LLM).
Ils servent à qualifier le corpus, pas à filtrer définitivement.

In [ ]:
def flag_noise(text):
    """Heuristiques inspirées de A3 (Musleh 2023) et A4 (Airlangga 2023)."""
    text = str(text).strip()
    if len(text) < 3:
        return "trop_court"
    if re.fullmatch(r"[^\w\s]+|[\d\W]+", text):
        return "emoji_only"
    if len(text.split()) == 1:
        return "reaction_vide"
    if re.search(r"https?://\S+|www\.\S+", text, re.IGNORECASE):
        return "contient_url"
    if re.search(r"(.)\1{4,}", text):
        return "lettres_repetees"
    return "ok"

df_clean["noise_flag"] = df_clean["text"].apply(flag_noise)
noise_dist = df_clean["noise_flag"].value_counts()
print(noise_dist.to_string())
ratio_bruit = 100 * (1 - noise_dist.get("ok", 0) / len(df_clean))
print(f"\nRatio de bruit estimé : {ratio_bruit:.1f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2ecc71" if k == "ok" else "#e74c3c" for k in noise_dist.index]
bars = ax.bar(noise_dist.index, noise_dist.values, color=colors, edgecolor="white")
ax.set_title("Flags de bruit heuristique (pré-pipeline A5)")
ax.set_ylabel("Nb commentaires")
for bar, val in zip(bars, noise_dist.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
        f"{val} ({100*val/len(df_clean):.1f}%)",
        ha="center", va="bottom", fontsize=9
    )
plt.tight_layout()
plt.savefig("../docs/phase2_data/fig_noise_flags.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Exemples de commentaires

In [ ]:
clean_sample = df_clean[df_clean["noise_flag"] == "ok"].copy()
print(f"Commentaires propres : {len(clean_sample)}")

if "author_likes" in clean_sample.columns:
    print("\nTop 5 les plus aimés :")
    print(clean_sample.nlargest(5, "author_likes")[["text", "author_likes"]].to_string())
else:
    print("\n5 exemples aléatoires :")
    print(clean_sample.sample(5, random_state=SEED)[["text"]].to_string())

In [ ]:
noisy_sample = df_clean[df_clean["noise_flag"] != "ok"]
if len(noisy_sample) > 0:
    print(f"Exemples de commentaires bruités ({len(noisy_sample)} total) :")
    print(noisy_sample.groupby("noise_flag").head(2)[["noise_flag", "text"]].to_string())
else:
    print("Aucun commentaire bruité détecté par les heuristiques.")

---
## 9. Résumé et export

In [ ]:
n_total  = len(df)
n_ok     = len(df_clean[df_clean["noise_flag"] == "ok"])
n_noisy  = len(df_clean) - n_ok
n_videos = df_clean["video_id"].nunique() if "video_id" in df_clean.columns else "N/A"

print("=" * 48)
print("RÉSUMÉ EDA — CORPUS DE COMMENTAIRES")
print("=" * 48)
print(f"Total brut              : {n_total:>8,}")
print(f"Propres (heuristique)   : {n_ok:>8,}  ({100*n_ok/n_total:.1f}%)")
print(f"Bruités (heuristique)   : {n_noisy:>8,}  ({100*n_noisy/n_total:.1f}%)")
print(f"Vidéos distinctes       : {str(n_videos):>8}")
print(f"Longueur moy (chars)    : {df_clean['char_length'].mean():>8.0f}")
print(f"Longueur méd (mots)     : {df_clean['word_count'].median():>8.0f}")
print("=" * 48)

In [ ]:
import pathlib
pathlib.Path("../data/clean").mkdir(parents=True, exist_ok=True)

export_cols = ["text", "noise_flag", "char_length", "word_count"]
for opt in ["video_id", "author_likes", "reply_count", "language"]:
    if opt in df_clean.columns:
        export_cols = [opt] + export_cols if opt == "video_id" else export_cols + [opt]

df_export = df_clean[[c for c in export_cols if c in df_clean.columns]]
df_export.to_csv(CLEAN_OUTPUT, index=False, encoding="utf-8")
print(f"Export : {CLEAN_OUTPUT} ({len(df_export)} lignes)")

---
## 10. Gold Standard — Annotation automatique

Le script `scripts/annotate_gold_standard.py` automatise les étapes suivantes :

| Étape | Méthode |
|---|---|
| Sélection stratifiée | 100 commentaires équilibrés (strates : bruit × longueur) |
| Annotation | 2 passages LLM indépendants (simule 2 annotateurs) |
| Kappa | Cohen's kappa sur `sentiment_label` — seuil 0.7 (AC-08 PRD) |
| Export | `data/gold_standard/annotated_videos.csv` |
| Tag Git | `v0.1.0` créé automatiquement si kappa > 0.7 |